In [ ]:
"""
ENHANCED BASELINE NOTEBOOK WITH VISION TRANSFORMER INTEGRATION
===============================================================

This notebook extends the baseline ResNet-50 with Vision Transformer (ViT)
to create a hybrid architecture for improved pneumothorax detection.

New additions are clearly marked with: # [VIT ENHANCEMENT]
Original baseline structure is preserved throughout.
"""

# ============================================================================
# CELL 1: IMPORTS (Enhanced with ViT requirements)
# ============================================================================

import numpy as np
import pandas as pd
import os
import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, hamming_loss, roc_curve, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*80)
print("HYBRID RESNET-50 + VISION TRANSFORMER MODEL")
print("="*80)
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("="*80)

In [ ]:
# ============================================================================
# CELL 2: [VIT ENHANCEMENT] - VISION TRANSFORMER CUSTOM LAYERS
# ============================================================================
"""
These custom layers enable Vision Transformer functionality:
1. PatchExtractor: Divides images into fixed-size patches
2. PatchEncoder: Adds positional information to patches
3. MultiHeadAttentionBlock: Applies self-attention for global feature learning

Why ViT for Medical Imaging?
- Captures long-range dependencies (e.g., pneumothorax affecting both lung fields)
- Learns spatial relationships across entire image
- Complements ResNet's local feature extraction
"""

class PatchExtractor(layers.Layer):
    """Extracts non-overlapping patches from input images"""
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches
    
    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


class PatchEncoder(layers.Layer):
    """Encodes patches with learnable position embeddings"""
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim
        })
        return config


class MultiHeadAttentionBlock(layers.Layer):
    """Transformer block with multi-head self-attention and MLP"""
    def __init__(self, projection_dim, num_heads, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.projection_dim = projection_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=projection_dim // num_heads,
            dropout=dropout_rate
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = keras.Sequential([
            layers.Dense(projection_dim * 2, activation=tf.nn.gelu),
            layers.Dropout(dropout_rate),
            layers.Dense(projection_dim),
            layers.Dropout(dropout_rate),
        ])

    def call(self, encoded_patches, training=False):
        # Attention with residual connection
        x1 = self.norm1(encoded_patches)
        attention_output = self.attn(x1, x1, training=training)
        x2 = layers.add([attention_output, encoded_patches])
        
        # MLP with residual connection
        x3 = self.norm2(x2)
        x3 = self.mlp(x3, training=training)
        output = layers.add([x3, x2])
        return output
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "projection_dim": self.projection_dim,
            "num_heads": self.num_heads,
            "dropout_rate": self.dropout_rate
        })
        return config

print("\n✓ Vision Transformer components loaded")
print("  - PatchExtractor: Divides images into patches")
print("  - PatchEncoder: Adds positional embeddings")
print("  - MultiHeadAttentionBlock: Self-attention mechanism")

In [ ]:
# ============================================================================
# CELL 3: CONFIGURATION (Enhanced for Hybrid Model)
# ============================================================================

# Dataset configuration (from baseline)
IMAGE_SIZE = 224
BATCH_SIZE = 32  # [VIT ENHANCEMENT] Reduced due to ViT memory requirements
EPOCHS = 60      # [VIT ENHANCEMENT] Adjusted for hybrid training
LEARNING_RATE = 0.0001  # [VIT ENHANCEMENT] Lower for stable training   

# [VIT ENHANCEMENT] Vision Transformer configuration
PATCH_SIZE = 16              # 16x16 patches
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 196 patches
PROJECTION_DIM = 256         # ViT embedding dimension
NUM_HEADS = 8                # Multi-head attention
TRANSFORMER_LAYERS = 4       # Number of transformer blocks
VIT_DROPOUT = 0.1           # ViT dropout rate

# [VIT ENHANCEMENT] Hybrid model configuration
RESNET_DROPOUT = 0.3        # ResNet dropout
FUSION_DROPOUT = 0.4        # Fusion layer dropout
FUSION_UNITS = 512          # Fusion layer size

print(f"\n{'='*80}")
print("CONFIGURATION")
print(f"{'='*80}")
print(f"\n📊 Dataset:")
print(f"  Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")

print(f"\n🔬 ResNet-50 Branch:")
print(f"  Pre-trained: ImageNet")
print(f"  Dropout: {RESNET_DROPOUT}")
print(f"  Output: 2048D features (local patterns)")

print(f"\n🎯 Vision Transformer Branch:")
print(f"  Patch Size: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"  Patches: {NUM_PATCHES}")
print(f"  Projection Dim: {PROJECTION_DIM}")
print(f"  Attention Heads: {NUM_HEADS}")
print(f"  Transformer Blocks: {TRANSFORMER_LAYERS}")
print(f"  Dropout: {VIT_DROPOUT}")
print(f"  Output: {PROJECTION_DIM}D features (global context)")

print(f"\n🔗 Feature Fusion:")
print(f"  Strategy: Concatenate ResNet + ViT features")
print(f"  Total Features: {2048 + PROJECTION_DIM}D")
print(f"  Fusion Units: {FUSION_UNITS}")
print(f"  Fusion Dropout: {FUSION_DROPOUT}")
print(f"{'='*80}")

In [ ]:
#============================================================================
# CELL 4: DATASET LOADING (Same as baseline)
# ============================================================================

BASE_DIR = "./dataset_pneumothorax/"
CSV_PATH = os.path.join(BASE_DIR, "balanced_dataset.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

print(f"\nLoading balanced Pneumothorax dataset...")

try:
    df = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded CSV with {len(df):,} total images")
except Exception as e:
    print(f"✗ Error reading CSV {CSV_PATH}: {e}")
    exit(1)

print(f"\n=== Dataset Structure ===")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# Use Binary_Label column
if 'Binary_Label' in df.columns:
    df['binary_label'] = df['Binary_Label']
elif 'Pneumothorax' in df.columns:
    df['binary_label'] = df['Pneumothorax'].astype(int)
else:
    print("✗ Could not find Pneumothorax labels")
    exit(1)

# Create full image paths
df['path'] = df['Image_Index'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Verify images exist
df['exists'] = df['path'].apply(os.path.exists)
missing_count = (~df['exists']).sum()

if missing_count > 0:
    print(f"\n⚠ Warning: {missing_count} images not found")
    df = df[df['exists']].copy()

df = df.drop(columns=['exists'])
balanced_df = df.copy()

print(f"\n=== Final Dataset Summary ===")
print(f"Total images: {len(df):,}")
print(f"Pneumothorax (Positive): {df['binary_label'].sum():,} ({df['binary_label'].sum()/len(df)*100:.1f}%)")
print(f"No Pneumothorax (Negative): {(len(df) - df['binary_label'].sum()):,} ({(len(df) - df['binary_label'].sum())/len(df)*100:.1f}%)")
print(f"Balance ratio: 1:1")
print(f"\n✓ Dataset is ready for training!")

In [ ]:
# ============================================================================
# CELL 5: DATASET SPLITTING (Same as baseline)
# ============================================================================

print(f"\nDataset ready for splitting: {len(balanced_df)} samples")

# Stratified train-validation-test split (80/10/10)
train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df['binary_label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['binary_label']
)

print(f"\n=== Dataset Splits (Stratified) ===")
print(f"Training set: {len(train_df):,} samples")
print(f"  Positive: {train_df['binary_label'].sum():,} ({train_df['binary_label'].sum()/len(train_df)*100:.1f}%)")
print(f"  Negative: {len(train_df) - train_df['binary_label'].sum():,} ({(len(train_df) - train_df['binary_label'].sum())/len(train_df)*100:.1f}%)")

print(f"\nValidation set: {len(val_df):,} samples")
print(f"  Positive: {val_df['binary_label'].sum():,} ({val_df['binary_label'].sum()/len(val_df)*100:.1f}%)")
print(f"  Negative: {len(val_df) - val_df['binary_label'].sum():,} ({(len(val_df) - val_df['binary_label'].sum())/len(val_df)*100:.1f}%)")

print(f"\nTest set: {len(test_df):,} samples")
print(f"  Positive: {test_df['binary_label'].sum():,} ({test_df['binary_label'].sum()/len(test_df)*100:.1f}%)")
print(f"  Negative: {len(test_df) - test_df['binary_label'].sum():,} ({(len(test_df) - test_df['binary_label'].sum())/len(test_df)*100:.1f}%)")

# Save splits
os.makedirs('./output', exist_ok=True)
test_df.to_csv('./output/pneumothorax_test_set_hybrid.csv', index=False)
print("\n✓ Test set saved for final evaluation")

In [ ]:
# ============================================================================
# CELL 6: DATA PIPELINE (Same as baseline with enhanced augmentation)
# ============================================================================

def create_tf_dataset(df, batch_size, shuffle=True, augment=True):
    """Enhanced tf.data pipeline with medical image augmentation"""
    
    def load_and_preprocess(image_path, label):
        # Load image
        img = tf.io.read_file(image_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
        
        # Enhanced augmentation for medical images
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
            img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            img = tf.image.random_brightness(img, max_delta=0.1)
            img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
            
            # Random zoom
            if tf.random.uniform([]) > 0.5:
                crop_size = tf.random.uniform([], 0.85, 1.0)
                h, w = IMAGE_SIZE, IMAGE_SIZE
                crop_h = tf.cast(h * crop_size, tf.int32)
                crop_w = tf.cast(w * crop_size, tf.int32)
                img = tf.image.random_crop(img, [crop_h, crop_w, 3])
                img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
        
        # Normalize using ResNet50V2 preprocessing
        img = tf.keras.applications.resnet_v2.preprocess_input(img)
        return img, label
    
    # Create dataset
    image_paths = df['path'].values
    labels = df['binary_label'].values.astype(np.float32)
    
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000, seed=42)
    
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create datasets
print("\nCreating tf.data pipelines for binary classification...")
train_dataset = create_tf_dataset(train_df, BATCH_SIZE, shuffle=True, augment=True)
val_dataset = create_tf_dataset(val_df, BATCH_SIZE, shuffle=False, augment=False)
test_dataset = create_tf_dataset(test_df, BATCH_SIZE, shuffle=False, augment=False)

print(f"✅ tf.data pipelines created:")
print(f"  Training batches: {len(train_dataset)}")
print(f"  Validation batches: {len(val_dataset)}")
print(f"  Test batches: {len(test_dataset)}")

# Verify dataset structure
for images, labels in train_dataset.take(1):
    print(f"\nDataset verification:")
    print(f"  Image batch shape: {images.shape}")
    print(f"  Label batch shape: {labels.shape}")
    print(f"  Label range: {tf.reduce_min(labels).numpy():.1f} to {tf.reduce_max(labels).numpy():.1f}")
    print(f"  Sample labels: {labels[:5].numpy()}")

In [ ]:
# ============================================================================
# CELL 7 (Refactored): [HYBRID ARCHITECTURE] RESNET-50 + VISION TRANSFORMER FUSION
# ============================================================================

def create_hybrid_resnet_vit_model(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    patch_size=PATCH_SIZE,
    num_patches=NUM_PATCHES,
    projection_dim=PROJECTION_DIM,
    num_heads=NUM_HEADS,
    transformer_layers=TRANSFORMER_LAYERS,
    vit_dropout=VIT_DROPOUT,
    resnet_dropout=RESNET_DROPOUT,
    fusion_dropout=FUSION_DROPOUT,
    fusion_units=FUSION_UNITS
):
    """
    Builds a hybrid deep model combining ResNet-50 (CNN) and Vision Transformer (ViT)
    for multi-level feature fusion.

    Returns:
        model (tf.keras.Model): Hybrid model combining CNN + Transformer
        resnet_base (tf.keras.Model): ResNet base (for unfreezing later)
    """

    # =========================================================================
    # INPUT LAYER
    # =========================================================================
    inputs = layers.Input(shape=input_shape, name='input_image')

    # =========================================================================
    # BRANCH 1: RESNET-50 (Local Feature Extraction)
    # =========================================================================
    resnet_base = ResNet50V2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        pooling="avg"
    )

    # Freeze early for stable hybrid training
    resnet_base.trainable = False

    resnet_features = resnet_base(inputs, training=False)
    resnet_features = layers.Dropout(resnet_dropout, name='resnet_dropout')(resnet_features)

    # =========================================================================
    # BRANCH 2: VISION TRANSFORMER (Global Feature Extraction)
    # =========================================================================
    # Step 1: Patch extraction
    patches = PatchExtractor(patch_size, name="vit_patch_extractor")(inputs)
    # Step 2: Patch encoding with positional embeddings
    encoded_patches = PatchEncoder(num_patches, projection_dim, name="vit_patch_encoder")(patches)

    # Step 3: Transformer encoder layers
    vit_features = encoded_patches
    for i in range(transformer_layers):
        vit_features = MultiHeadAttentionBlock(
            projection_dim=projection_dim,
            num_heads=num_heads,
            dropout_rate=vit_dropout,
            name=f"vit_transformer_block_{i}"
        )(vit_features)

    # Step 4: Global average pooling
    vit_features = layers.GlobalAveragePooling1D(name="vit_global_pool")(vit_features)
    vit_features = layers.Dropout(vit_dropout, name="vit_dropout")(vit_features)

    # =========================================================================
    # FUSION BLOCK: Adaptive Feature Fusion
    # =========================================================================
    # Project both branches to a common dimensional space
    resnet_proj = layers.Dense(fusion_units, activation='relu', name='resnet_projection')(resnet_features)
    vit_proj = layers.Dense(fusion_units, activation='relu', name='vit_projection')(vit_features)

    # Combine features using learnable fusion weights (instead of plain concat)
    fused = layers.Concatenate(name='fusion_concat')([resnet_proj, vit_proj])
    fused = layers.Dense(fusion_units, activation='relu', name='fusion_dense')(fused)
    fused = layers.BatchNormalization(name='fusion_bn')(fused)
    fused = layers.Dropout(fusion_dropout, name='fusion_dropout')(fused)

    # =========================================================================
    # CLASSIFICATION HEAD
    # =========================================================================
    outputs = layers.Dense(
        1,
        activation='sigmoid',
        kernel_regularizer=tf.keras.regularizers.l2(0.01),
        name='pneumothorax_prediction'
    )(fused)

    # =========================================================================
    # FINAL MODEL
    # =========================================================================
    model = keras.Model(inputs=inputs, outputs=outputs, name="Hybrid_ResNet50_ViT")

    return model, resnet_base


# ============================================================================
# MODEL BUILD SUMMARY
# ============================================================================
print("\n" + "="*80)
print("BUILDING HYBRID RESNET-50 + VISION TRANSFORMER (FUSION MODEL)")
print("="*80)

model, resnet_base = create_hybrid_resnet_vit_model()

print(f"\n📊 Model Summary:")
print(f"  Total parameters: {model.count_params():,}")
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Non-trainable parameters: {model.count_params() - trainable_params:,}")
print(f"  ResNet base frozen: {not resnet_base.trainable}")

print(f"\n💡 Expected Improvements:")
print(f"  ✓ Global context from ViT complements local texture from ResNet")
print(f"  ✓ Adaptive fusion improves integration of feature hierarchies")
print(f"  ✓ Anticipated AUC improvement: +3–6%")
print("="*80)


In [ ]:
# ============================================================================
# CELL 8: CLASS WEIGHTS (Same as baseline)
# ============================================================================

from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['binary_label']),
    y=train_df['binary_label']
)
class_weights = dict(enumerate(class_weights_array))

print(f"\n⚖️ Class Weights:")
print(f"  Negative (0): {class_weights[0]:.3f}")
print(f"  Positive (1): {class_weights[1]:.3f}")

In [ ]:
# ============================================================================
# CELL 9: MODEL COMPILATION
# ============================================================================

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')
    ]
)

print("\n✅ Model Compiled")
print(f"  Optimizer: Adam (lr={LEARNING_RATE})")
print(f"  Loss: Binary Cross-Entropy")
print(f"  Metrics: AUC, Accuracy, Precision, Recall")

In [ ]:
# ============================================================================
# CELL 10: CALLBACKS
# ============================================================================

callbacks = [
    ModelCheckpoint(
        filepath='./output/hybrid_resnet_vit_best.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    EarlyStopping(
        monitor='val_auc',
        patience=10,
        mode='max',
        restore_best_weights=True,
        verbose=1,
        min_delta=0.001
    ),
    
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
        min_delta=0.0001
    )
]

print("\n✅ Callbacks Configured")

In [ ]:
# ============================================================================
# CELL 11: [VIT ENHANCEMENT] THREE-STAGE TRAINING
# ============================================================================

print("\n" + "="*80)
print("THREE-STAGE HYBRID TRAINING STRATEGY")
print("="*80)
print("\nStage 1: Train ViT + Fusion (ResNet frozen) - 15 epochs")
print("Stage 2: Fine-tune ResNet top + ViT - 20 epochs")
print("="*80 + "\n")

# ========================================================================
# STAGE 1: Train ViT branch
# ========================================================================
print("\n" + "="*80)
print("STAGE 1: Training ViT Branch (ResNet Frozen)")
print("="*80)
print(f"ResNet trainable: {resnet_base.trainable}")
print(f"Learning rate: {LEARNING_RATE}")
print("="*80 + "\n")

history_stage1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\n✅ Stage 1 Complete!")

# ========================================================================
# STAGE 2: Unfreeze ResNet top layers for joint fine-tuning
# ========================================================================
print("\n" + "="*80)
print("STAGE 2: Fine-tuning ResNet + ViT")
print("="*80)

# Unfreeze last 30 layers
print("Unfreezing ResNet top layers...")
for layer in resnet_base.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

total_layers = len(resnet_base.layers)
trainable_layers = sum([layer.trainable for layer in resnet_base.layers])

print(f"  ResNet total layers: {total_layers}")
print(f"  ResNet trainable layers: {trainable_layers}")
print(f"  ResNet frozen layers: {total_layers - trainable_layers}")

trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"  Total trainable parameters: {trainable_params:,}")

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE * 0.1),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')
    ]
)

print(f"\nModel recompiled:")
print(f"  New learning rate: {LEARNING_RATE * 0.1}")
print("="*80 + "\n")

history_stage2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    initial_epoch=10,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\n✅ Stage 2 Complete!")

# Combine histories
history = history_stage1
for key in history_stage2.history.keys():
    history.history[key].extend(history_stage2.history[key])

# Save final model
model.save('./output/hybrid_resnet_vit_final.keras')

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print("Models saved:")
print("  ✓ Best model: ./output/hybrid_resnet_vit_best.keras")
print("  ✓ Final model: ./output/hybrid_resnet_vit_final.keras")
print("="*80)

In [ ]:
# ============================================================================
# TRAINING HISTORY VISUALIZATION
# ============================================================================

def plot_training_history(history):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Training Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # AUC
    axes[0, 1].plot(history.history['auc'], label='Training AUC')
    axes[0, 1].plot(history.history['val_auc'], label='Validation AUC')
    axes[0, 1].set_title('Model AUC')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AUC')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Training Precision')
    axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
    axes[1, 0].set_title('Model Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Training Recall')
    axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
    axes[1, 1].set_title('Model Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./output/baseline_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION - BINARY CLASSIFICATION
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION")
print("="*80)

# Generate predictions on validation dataset
print("Generating predictions on validation set...")
val_predictions = model.predict(val_dataset, verbose=1)

# Get true labels from validation dataframe - FIXED
y_true = val_df['binary_label'].values  # Use binary_label column
y_pred_probs = val_predictions.flatten()  # Flatten to 1D array
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

print(f"\nPredictions shape: {y_pred_probs.shape}")
print(f"True labels shape: {y_true.shape}")
print(f"Unique predictions: {np.unique(y_pred)}")
print(f"Unique true labels: {np.unique(y_true)}")

# Calculate metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# AUC Score
auc_score = roc_auc_score(y_true, y_pred_probs)
print(f"\n📊 VALIDATION METRICS:")
print(f"  AUC Score: {auc_score:.4f}")

# Classification Report
print(f"\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=['No Pneumothorax', 'Pneumothorax']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\n🔢 Confusion Matrix:")
print(f"  True Negatives: {cm[0,0]}")
print(f"  False Positives: {cm[0,1]}")
print(f"  False Negatives: {cm[1,0]}")
print(f"  True Positives: {cm[1,1]}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # Recall
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
f1_score = 2 * (precision * sensitivity) / (precision + sensitivity)

print(f"\n📈 Detailed Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  F1-Score: {f1_score:.4f}")
print(f"  Accuracy: {(tp + tn) / (tp + tn + fp + fn):.4f}")

In [ ]:
# ============================================================================
# VISUALIZE PREDICTIONS - FIXED
# ============================================================================

def visualize_predictions(test_df, y_pred_probs, n_samples=12):
    """Visualize sample predictions"""
    # Get sample indices
    indices = np.random.choice(len(test_df), n_samples, replace=False)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    for idx, img_idx in enumerate(indices):
        img_path = test_df.iloc[img_idx]['path']
        true_label = test_df.iloc[img_idx]['binary_label']  # FIXED: Use 'binary_label'
        pred_prob = y_pred_probs[img_idx]  # FIXED: Remove [0] since it's already 1D
        pred_label = 1 if pred_prob > 0.5 else 0
        
        # Load and display image
        img = keras.preprocessing.image.load_img(img_path, target_size=(224,224))
        img_array = keras.preprocessing.image.img_to_array(img)
        
        axes[idx].imshow(img_array.astype('uint8'))
        
        # Create title with prediction info
        true_class = "Pneumothorax" if true_label == 1 else "Negative"
        pred_class = "Pneumothorax" if pred_label == 1 else "Negative"
        color = 'green' if true_label == pred_label else 'red'
        
        title = f"True: {true_class}\nPred: {pred_class} ({pred_prob:.3f})"
        axes[idx].set_title(title, color=color, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('./output/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

# Get predictions on test set first
print("Generating predictions on test set...")
test_predictions = model.predict(test_dataset, verbose=1)
test_pred_probs = test_predictions.flatten()  # Already 1D

print(f"Test predictions shape: {test_pred_probs.shape}")
print(f"Test dataframe shape: {test_df.shape}")

# Visualize predictions
visualize_predictions(test_df.reset_index(drop=True), test_pred_probs)

In [ ]:
# Load best model

# Evaluate on val and test
val_metrics = model.evaluate(val_dataset)
test_metrics = model.evaluate(test_dataset)

print(f"Val AUC: {val_metrics}")
print(f"Test AUC: {test_metrics}")